# Notebook 3 : stabilité et condition CFL

**Question** : que se passe-t-il si le pas de temps est trop grand ?

On reprend le test de la tache qui avance (notebook 2), avec le schéma upwind + Euler explicite, et on ne change qu'une chose : le nombre **CFL**. Le pas de temps est `dt = CFL / max(somme des vitesses sortantes / dx)`. Le schéma est stable pour **CFL ≤ 1**.

Les fonctions du solveur sont dans le fichier `advection.py` (même dossier que ce notebook). On les importe, et on peut les lire en ouvrant ce fichier.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.makedirs('figures', exist_ok=True)
from advection import *

## 1. Le cas test

On utilise un **créneau** (carré de concentration 1) plutôt qu'une gaussienne : il contient toutes les fréquences, donc les oscillations parasites apparaissent vite. Avec une gaussienne lisse, une instabilité peut mettre très longtemps à se voir, parce qu'elle part du bruit d'arrondi (de l'ordre de 1e-16).

Domaine de 200 x 100, 100 cellules en x (dx = 2), vitesse 1 vers la droite, durée 100.

In [ ]:
liste_cfl = [0.5, 0.9, 1.0, 1.1, 1.5]
resultats = {}
for cfl in liste_cfl:
    resultats[cfl] = cas_translation(N=100, profil='creneau', schema='upwind', temps='euler',
                                     cfl=cfl, t_end=100.0, Lx=200.0)

print(f"{'CFL':>5} {'dt':>7} {'pas':>5} {'max|c| max':>12} {'explose ?':>10}")
for cfl, r in resultats.items():
    print(f"{cfl:5.2f} {r['dt']:7.3f} {r['n']:5d} {r['cmax'].max():12.4g} {str(r['explose']):>10}")

Le calcul s'arrête automatiquement si max|c| dépasse 1e6 (sinon on obtiendrait des `inf` et des `nan`).

## 2. Le maximum de |c| au cours du temps

In [ ]:
plt.figure(figsize=(8, 5))
for cfl, r in resultats.items():
    plt.plot(r['temps'], r['cmax'], label=f"CFL = {cfl}")
plt.yscale('log')
plt.axhline(1, color='k', lw=0.5)
plt.xlabel("t"); plt.ylabel("max |c|")
plt.title("Upwind + Euler explicite : effet du CFL")
plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.savefig('figures/stabilite_cfl.png', dpi=200, bbox_inches='tight')
plt.show()

**À observer** :
- CFL < 1 : max|c| ne dépasse jamais 1 (il diminue : l'upwind étale le créneau).
- CFL = 1 : max|c| reste exactement 1 : dans ce cas simple, la tache se déplace d'une cellule exactement par pas, c'est la solution exacte.
- CFL > 1 : max|c| croît, de plus en plus vite quand CFL augmente. C'est l'instabilité.

## 3. La tache à la fin du calcul

In [ ]:
fig, axs = plt.subplots(len(liste_cfl), 1, figsize=(10, 2.2 * len(liste_cfl)))
for ax, (cfl, r) in zip(axs, resultats.items()):
    im = ax.imshow(r['c'], vmin=-0.5, vmax=1.5, cmap='RdBu_r')
    ax.set_title(f"CFL = {cfl}   (t = {r['t_final']:.0f}, max c = {r['c'].max():.3g}, min c = {r['c'].min():.3g})", fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.savefig('figures/stabilite_images.png', dpi=200, bbox_inches='tight')
plt.show()

Le fond (c = 0) apparaît en bleu moyen, la tache saine en rose. Pour CFL > 1, les bandes **rouge foncé** (c très positif) et **bleu foncé** (c très négatif) sont des oscillations de plus en plus fortes derrière le front : des valeurs de c négatives ou supérieures à 1, qui n'ont aucun sens physique (regarde les valeurs max et min dans les titres).

## 4. Pourquoi ? (analyse de Von Neumann, facultatif)

Sur une grille 1D, avec ν = CFL, le schéma upwind s'écrit : c_j^(n+1) = (1 − ν) c_j^n + ν c_(j−1)^n. C'est une moyenne pondérée tant que 0 ≤ ν ≤ 1 : le résultat reste entre le min et le max de départ. Pour ν > 1, un coefficient devient négatif et le schéma amplifie les erreurs.

En cherchant une onde c_j^n = G^n exp(i j θ), on trouve le facteur d'amplification : **G = 1 − ν (1 − e^(−iθ))**, soit |G|² = 1 − 2ν(1 − ν)(1 − cos θ).
- |G| ≤ 1 pour tout θ si et seulement si ν ≤ 1 : condition CFL.
- Le mode le plus amplifié est θ = π (damier), avec |G| = |1 − 2ν|.

In [ ]:
theta = np.linspace(0, np.pi, 400)
plt.figure(figsize=(8, 4.5))
for nu in [0.5, 1.0, 1.1, 1.5]:
    G = np.sqrt(1 - 2 * nu * (1 - nu) * (1 - np.cos(theta)))
    plt.plot(theta, G, label=f"upwind, ν = {nu}")
plt.plot(theta, np.sqrt(1 + 0.5**2 * np.sin(theta)**2), '--', label="centré + Euler, ν = 0.5")
plt.axhline(1, color='k', lw=0.5)
plt.xlabel("θ (nombre d'onde adimensionné)"); plt.ylabel("|G|")
plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig('figures/von_neumann.png', dpi=200, bbox_inches='tight')
plt.show()

Les courbes au-dessus de 1 correspondent à des modes qui grandissent à chaque pas. Le schéma centré + Euler (pointillés) est **toujours au-dessus de 1**, même pour un petit ν : il est instable quel que soit le pas de temps. C'est l'objet de la section suivante.

## 5. Schéma centré + Euler explicite : instable quel que soit le CFL

Le schéma centré prend `c_face = (c_gauche + c_droite)/2`. Il est d'ordre 2 en espace, mais associé à Euler explicite il est instable pour tout CFL. Cela montre que **la stabilité dépend du couple schéma en espace / schéma en temps**, pas seulement de la taille de dt.

In [ ]:
plt.figure(figsize=(8, 5))
for cfl in [0.1, 0.5]:
    r = cas_translation(N=100, profil='creneau', schema='centre', temps='euler',
                        cfl=cfl, t_end=100.0, Lx=200.0)
    plt.plot(r['temps'], r['cmax'], label=f"centré + Euler, CFL = {cfl}")
r = resultats[0.5]
plt.plot(r['temps'], r['cmax'], 'k--', label="upwind + Euler, CFL = 0.5")
plt.yscale('log'); plt.xlabel("t"); plt.ylabel("max |c|")
plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.savefig('figures/centre_instable.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. Conclusion à rédiger

- À partir de quel CFL le calcul explose-t-il ? Est-ce cohérent avec la théorie (CFL = 1) ?
- Pourquoi CFL = 1 donne-t-il la solution exacte dans ce cas particulier (vitesse uniforme alignée sur la grille) et pas dans le labyrinthe ?
- Que se passe-t-il pour le schéma centré, même avec un petit CFL ?
- Dans le labyrinthe, les vitesses varient d'une cellule à l'autre : quelle est la cellule qui impose le pas de temps ?